In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from robusta_hmf import Robusta
import time
#%matplotlib widget

In [ ]:
#define constants
c = 299792.458 # km/s
LN10 = np.log(10.)
MAX_IVAR = 2.5e3
MIN_IVAR = 0.1
H_ALPHA, H_BETA = 6564.614, 4862.721 #Reference: from classic.sdss.org
HE_I_6680 = 6680 #Reference: Johanna brain
HE_I_4026 = 4026
HE_I_4471 = 4471
HE_I_4922 = 4922
HE_I_4713 = 4713
HE_II_4686 = 4686 #Reference: Johanna brain
HE_II_4542 = 4542
HE_II_4200 = 4200


#lines
lines_balmer = [H_ALPHA, H_BETA]
lines_HE_I = [HE_I_6680, HE_I_4026, HE_I_4471, HE_I_4922, HE_I_4713]
lines_HE_II = [HE_II_4686, HE_II_4542, HE_II_4200]
all_lines = np.concatenate([lines_balmer, lines_HE_I, lines_HE_II])
mask_lines = lines_balmer

#line labes
line_labels_balmer = ["Halpha", "Hbeta"]
line_labels_HE_I = ["HeI_6680", "HeI_4026", "HeI_4471", "HeI_4922", "HeI_4713"]
line_labels_HE_II = ["HeII_4686", "HeII_4542", "HeII_4200"]
all_line_labels = np.concatenate([line_labels_balmer, line_labels_HE_I, line_labels_HE_II])

statistics = ["EW", "shift", "width"]
moment_names = ["EW", "M1", "M2", "M3", "M4"]


H_delta_lnlam_line = 1000 / c 
H_delta_loglam_line = H_delta_lnlam_line / LN10 
HE_delta_lnlam_line = 500 / c
HE_delta_loglam_line = HE_delta_lnlam_line / LN10
# log_H_ALPHA, log_H_BETA, log_HE_I, log_HE_II = np.log10(H_ALPHA), np.log10(H_BETA), np.log10(HE_I), np.log10(HE_II)

#delta loglam line for labeling
delta_balmer = np.zeros(len(lines_balmer)) + (1000 / c) / LN10 
delta_HE_I = np.zeros(len(lines_HE_I)) + (500 / c) / LN10 
delta_HE_II = np.zeros(len(lines_HE_II)) + (500 / c) / LN10 
all_delta_loglams = np.concatenate([delta_balmer, delta_HE_I, delta_HE_II])


#indexing is super important for these three arrays 
print(all_lines.shape, all_line_labels.shape, all_delta_loglams.shape)
print(all_lines, all_line_labels, all_delta_loglams)

H_delta_lnlam_line = 1000 / c 
H_delta_loglam_line = H_delta_lnlam_line / LN10 
HE_delta_lnlam_line = 500 / c
HE_delta_loglam_line = HE_delta_lnlam_line / LN10
log_H_ALPHA, log_H_BETA = np.log10(H_ALPHA), np.log10(H_BETA)


half_wids = [H_delta_loglam_line, H_delta_loglam_line, HE_delta_loglam_line, HE_delta_loglam_line] 

#THIS MUST BE ASCENDING ORDER
temp_cuts = np.array([10000, 15000, 25000])

In [ ]:
print(all_lines)

In [ ]:
#load stellar parameters
DATESTR = "2026-07-20"

stars = pd.read_parquet(f"parent_stars10k_{DATESTR}.parquet")
print("num of stars in parquet:", stars.shape[0])

spectra_temp = pd.read_parquet(f"spectra_{DATESTR}_table.parquet")
print("num of spectra in parquet:", stars.shape[0])

print(stars.shape)

In [ ]:
print("spectra before:", spectra_temp.shape)
spectra = spectra_temp.join(stars[["GAIA_ID", "Teff_fit", "NANA_HASH"]].set_index("GAIA_ID"), on = "GAIA_ID", rsuffix = "_stars", how = "left", validate = "m:1")
print("spectra after:", spectra.shape)

In [ ]:
nana = "nana_"
count = 0
for l, line in enumerate(all_line_labels):
    col_name = nana + line + "_"
    print(col_name)
    for s, stat in enumerate(moment_names):
        ##add column here
            #print(col_name + stat)
            spectra[col_name + stat] = 0.
            count+=1
            spectra[col_name + stat + "_err"] = np.inf
            count+=1
            # spectra[col_name + stat + "_resid"] = 0. ASK HOGG ABOUT THIS, JOHANNA WANTED EW MEASUREMENTS OF THE RESID
            # spectra[col_name + stat + "_resid_err"] = 0.

print(spectra.columns.to_list())
print("You added", count, "amount of columns. Does it match (num of lines) * (num of moments) * (2)?")

In [ ]:
# sanity check contents of file
print(spectra[["GAIA_ID", "NANA_HASH", "Teff_fit", "EWobs_Halpha", "rv_hydrogen_all_mean", "SPEC_FILE"]])

In [ ]:
#read in the data and deal with radial velocities

#take the ALL spectra (like 35,000)
with open(f'spectra_{DATESTR}_data.pkl','rb') as f:
    pkl_data = pickle.load(f)
print(len(pkl_data))

In [ ]:
#pkl of ALL spectra (Teff > 10000)
# BUG: This code is very brittle
fluxes, loglam, ivars, continuua, _, _, _, _, _ = pkl_data
print(fluxes.shape, loglam.shape, ivars.shape, continuua.shape)

In [ ]:
# hard stop if we SUCK
assert len(fluxes) == len(spectra)

In [ ]:
# shift to rest frame.
# Note: Only does integer-pixel shifts
# Note: Data have been slightly corrupted; delta_log_lam is not consistent across the wavelength grid.

def get_delta_log_lam(loglambdas):
    return np.median(loglambdas[1:] - loglambdas[:-1])
    
def pixel_shift(flxs, ivrs, contins, rvs, log_lambda):

    delta_log_lambda_pixel = get_delta_log_lam(log_lambda)

    # Catch bad rv values, goddammit
    bad = np.logical_not(np.isfinite(rvs))
    rad_vels = rvs.copy()
    rad_vels[bad] = 0.

    delta_log_lambdas = (rad_vels / c) / LN10
    delta_pixels = np.round(delta_log_lambdas/delta_log_lambda_pixel).astype(int)

    rest_flxs = np.zeros_like(flxs) + 1.
    rest_ivrs = np.zeros_like(ivrs)
    rest_contins = np.zeros_like(contins) + np.nan

    for i, dp in enumerate(delta_pixels):
        if dp < 0:
            rest_flxs[i, -dp:] = flxs[i, :dp]
            rest_ivrs[i, -dp:] = ivrs[i, :dp]
            rest_contins[i, -dp:] = contins[i, :dp]
            
        elif dp > 0:
            rest_flxs[i, :-dp] = flxs[i, dp:]
            rest_ivrs[i, :-dp] = ivrs[i, dp:]
            rest_contins[i, :-dp] = contins[i, dp:]
            
        else:
            rest_flxs[i, :] = flxs[i, :]
            rest_ivrs[i, :] = ivrs[i, :]
            rest_contins[i, :] = contins[i, :]

    # dammit
    rest_ivrs[bad, :] = 0.
    print("zeroing out ivars on", np.sum(bad), "stars with bad rvs")
    return rest_flxs, rest_ivrs, rest_contins

In [ ]:
# shift everything to the rest frame
rest_fluxes, rest_ivars, rest_continuua = pixel_shift(fluxes, ivars, continuua, spectra["rv_hydrogen_all_mean"].to_numpy(), loglam)
print(rest_fluxes.shape)

In [ ]:
def fix_nans_and_infinites(rest_flxs, rest_ivrs):

    #fix nans and infinities
    bad = np.logical_not(np.isfinite(rest_flxs))
    rest_flxs[bad] = 1
    rest_ivrs[bad] = 0
    
    bad = np.logical_not(np.isfinite(rest_ivrs))
    rest_flxs[bad] = 1
    rest_ivrs[bad] = 0
    
    bad = np.logical_or((rest_flxs > 2.0), (rest_flxs < 0))
    rest_flxs[bad] = 1
    rest_ivrs[bad] = 0
    
    bad = rest_ivrs > MAX_IVAR
    # rest_fluxes[bad] = 1
    # rest_ivars[bad] = 0
    rest_ivrs[bad] = MAX_IVAR
    
    bad = rest_ivrs < MIN_IVAR
    rest_flxs[bad] = 1.0
    rest_ivrs[bad] = MIN_IVAR

    return rest_flxs, rest_ivrs

In [ ]:
data, weights = fix_nans_and_infinites(rest_fluxes, rest_ivars)
print(data.shape, weights.shape, rest_continuua.shape)

In [ ]:
# make a sanity plot
tempposts = np.exp(np.linspace(np.log(np.min(spectra["Teff_fit"].to_numpy())), np.log(np.max(spectra["Teff_fit"].to_numpy())), 16))
f = plt.figure(figsize=(12, 12))
for t, T in enumerate(tempposts):
    # find closest high SNR spectrum
    highsnr = np.arange(len(spectra))[np.median(rest_ivars[:, :2000], axis=1) > 500.0]
    ii = np.argmin(np.abs(spectra["Teff_fit"].to_numpy()[highsnr] - T))
    ii = highsnr[ii]
    Teff = spectra["Teff_fit"].iloc[ii]
    print(t, T, ii, Teff)
    # plot it
    plt.step(10. ** loglam, rest_fluxes[ii] + t, c="k", where="mid")
    gid = spectra["GAIA_ID"].iloc[ii]
    label = f"{gid}; Teff = {Teff:5.0f}"
    plt.text(4000., 1.1 + t, label)
plt.ylim(0., len(tempposts) + 1.)
plt.xlim(4000, 5000)
#plt.xlim(H_ALPHA - 200, H_ALPHA + 200)

lines_balmer = [H_ALPHA, H_BETA]
lines_HE_I = [HE_I_6680, HE_I_4026, HE_I_4471, HE_I_4922, HE_I_4713]
lines_HE_II = [HE_II_4686, HE_II_4542, HE_II_4200]

for l, line in enumerate(lines_balmer):
    if l == 0:
        plt.axvline(line, label = "Balmer lines", lw=1, alpha=0.5, zorder=-10, color = "green")
    else:
        plt.axvline(line, lw=1, alpha=0.5, zorder=-10, color = "green")
        
for l, line in enumerate(lines_HE_I):
    if l == 0: 
        plt.axvline(line, label = "HeI lines", lw=1, alpha=0.5, zorder=-10, color = "orange")
    else:
        plt.axvline(line, lw=1, alpha=0.5, zorder=-10, color = "orange")
        
for l, line in enumerate(lines_HE_II):
    if l == 0: 
        plt.axvline(line, label = "HeII lines", lw=1, alpha=0.3, zorder=-10, color = "red")
    
    plt.axvline(line, lw=1, alpha=0.3, zorder=-10, color = "red")

plt.legend()    
plt.title("selected high SNR example stars, ordered by Teff")

In [ ]:
def make_temp_binning(temp_ranges, rest_flxs, rest_ivrs, rest_contins, spectra_df):

    #cut on Johanna's and Nanas' ews
    ews_j = spectra_df['EWobs_Halpha'].to_numpy()
    ews_n = spectra_df['nana_Halpha_EW'].to_numpy()
    good_ew_mask = (ews_j > 0) & (ews_n < 1)
    print(f"Johanna and Nana EW: Keeping {good_ew_mask.sum()} / {len(good_ew_mask)} spectra (removed {(~good_ew_mask).sum()})")
    rest_flxs, rest_ivrs, rest_contins, nana_hash, spectra_temps = rest_flxs[good_ew_mask], rest_ivrs[good_ew_mask], rest_contins[good_ew_mask], spectra_df["NANA_HASH"].to_numpy()[good_ew_mask], spectra_df["Teff_fit"].to_numpy()[good_ew_mask]



    ret_vals = {}
    labels = np.arange(0, len(temp_ranges), 1)
    print(labels)
    for label, temp in zip(labels, temp_ranges):
        good_temp_mask = temp <= spectra_temps

        
        ret_vals[label] = {
            "temp cut": temp,
            "flux": rest_flxs[good_temp_mask],
            "ivar": rest_ivrs[good_temp_mask],
            "continuua": rest_contins[good_temp_mask],
            "NANA_HASH": nana_hash[good_temp_mask],
            "Teff": spectra_temps[good_temp_mask],
            "count": int(good_temp_mask.sum()),
        }
        
        print(f"Number of spectra with temp >= {temp} is {ret_vals[label]['count']}")
    return ret_vals

In [ ]:
data.shape

In [ ]:
temp_binning_data = make_temp_binning(temp_cuts, data, weights, rest_continuua, spectra)

print("num of temp cuts:", len(temp_binning_data))

In [ ]:
# split data to A and B using nana_hash parity, only train on A
def split_and_train(temp_dict, K = 12, scale = 1, nu = 1):
    

    model_dict = {}
    mod_list = [0,1]
    for key, value in temp_dict.items():

        rest_flxs = value["flux"]
        rest_ivrs = value["ivar"]
        n_hash = value["NANA_HASH"]
        temp_cut = value["temp cut"]
        N, M = rest_flxs.shape

        for m in mod_list:
            inx = np.where(n_hash % 2 == m)[0]

            print(f" for mod {m} there are {len(inx)} spectra") 
            
            model = Robusta(rank=K, robust=True, robust_scale = scale, robust_nu = nu)
            start = time.perf_counter()
            model.fit(rest_flxs[inx], rest_ivrs[inx], max_iter=10000)
            end = time.perf_counter()
            print("total time:", (end - start)/60, "minutes")
        
            model_dict[m, temp_cut] = {"temp_cut": temp_cut, "mod": m, "model": model, "indx": inx}


    return model_dict

In [ ]:
models = split_and_train(temp_binning_data)
print(len(models))

In [ ]:
#always keep data the same shape
#compute temp binning and mods at the same time

#set up indexes, a training index and a syntheiszed index

In [ ]:
Mod_list = [0,1]
temp_cuts = np.asarray(temp_cuts, dtype=float)

temp_cuts = [10000, 15000, 25000, np.inf]
def get_synth_for_every_spectra(spectra_df, model_dict, mod_list, temp_ranges, temp_dict, lines, log_lambdas):

    #masking Halpha and Hbeta
    log_lines = np.log10(lines)
    log_H_ALPHA, log_H_BETA = log_lines[0], log_lines[1]
    region_alpha = np.abs(log_lambdas - log_H_ALPHA)
    region_beta = np.abs(log_lambdas - log_H_BETA)
    censor_mask = np.ones_like(log_lambdas)
    censor_mask[(region_alpha < H_delta_loglam_line) | (region_beta < H_delta_loglam_line) ] = 0 #?

    temp_ranges = temp_ranges[:-1]

    synth_dict = {}
    synth = np.zeros_like(rest_fluxes) + np.nan
    print("making synth:", synth.shape)
    
    for label, temp in enumerate(temp_ranges): #recall integer steps as label in the temp_dict
        print("synthesized for exclusive bin of temp:", temp)
        rest_fluxes = temp_dict[label]["flux"]
        print("data shape:", rest_fluxes.shape)
        rest_ivars = temp_dict[label]["ivar"]
        teffs_at_tempcut = temp_dict[label]["Teff"]
        nana_hash = temp_dict[label]["NANA_HASH"]

        #TRICKY PART, to identify which binning to use
        #example if temp cuts where [10000, 15000, 25000], and Teff for a certain spectrum was 16000
        #we need the correpsonding label in the Teff >= 150000, np.searchsorted(right) will return the index
        #if it was theoreitcally inserted into the temp_cuts ex: [10000, 15000, 16000, 25000] ==> 2
        #side = 'right' ensures that if Teff = 10000, it would be inserted to the right
        
        sindx = np.searchsorted(temp_ranges, teffs_at_tempcut, side = 'right')
        correct_temp_label = sindx - 1 #do you get why - 1 ? if Teff [15000, 16000, 17000, 28000] you would get [1,1,1,2] - 1 ==> [0,0,0,1]
        exclusive_tempbin_mask = correct_temp_label == label #ex ==> true, true, true, false
        for mod in mod_list:
        
            key = (label, mod)
            model = model_dict[key]["model"]
            
            #we want the OTHER indices that correspond with the spectra for synthesis
            assert mod_list == [0,1]
            other_mod = 1 -  mod #this is bad bad idea, only works when mod_list = [0,1]
            print("Trained on mod:", mod, "synthesizing on mod:", other_mod)
            key = (correct_temp_label, other_mod)
            synth_indx = (nana_hash%2 == other_mod) & (exclusive_tempbin_mask)
            state, _ = model.infer(rest_fluxes[synth_indx], rest_ivars[synth_indx] * censor_mask[None, :])
            synth[synth_indx] = model.synthesize(state)
            print("TEMP:", temp)
            synth_dict[label, mod] = {"temp": temp, "synth_spec": synth, "synth_index": synth_indx}

        
    return synth_dict, synth

In [ ]:
synthesized, synth_spectra = get_synth_for_every_spectra(spectra, models, Mod_list, temp_cuts, temp_binning_data, mask_lines, loglam)

In [ ]:
print(synth_spectra.shape, np.sum(np.isfinite(synth_spectra)))

In [ ]:
example_temp_cut = synthesized[2,0]["synth_spec"]
example_index = synthesized[2,0]["synth_index"]
#print(np.isnan(example_temp_cut))
print(example_temp_cut.shape)
print(example_index.shape)
print(example_temp_cut[example_index].shape)

In [ ]:
def get_moment_vals(log_lambda, rest_flxes, rest_ivrs, synth_indx, synth, lines):
    
    delta_log_lambda_pixel = get_delta_log_lam(log_lambda)
    N, M = rest_flxes[synth_indx].shape


    lam = 10 ** log_lambda

    exponents = np.arange(5)
    moments = np.zeros((N, len(lines), len(exponents))) + np.nan
    moment_errs = np.zeros((N, len(lines), len(exponents))) + np.nan

    for j, (line, wid) in enumerate(zip(lines, half_wids)):
        logline = np.log10(line)
        integration_weight = delta_log_lambda_pixel * LN10 * lam * (np.abs(logline - log_lambda) < wid)
        resid = rest_flxes[synth_indx] - synth[synth_indx]
        diff = lam - line #this is hogg's lam-lam0 he wrote about
    
        #compute vals and corresponding errors
        for k, expo in enumerate(exponents):
            something = integration_weight * diff ** expo
            moments[:, j, k] = np.sum(resid * something[None, :], axis=1)
            moment_errs[:, j, k] = np.sqrt(np.sum(something[None, :] ** 2 / rest_ivrs[synth_indx], axis=1))

    return moments, moment_errs

In [ ]:
print(len(synthesized[0]["synth_index"]))
print(len(synthesized[0]["synth_spec"]))
print(len(synthesized[0]["exclusive_tempbin"]))
print(len(synthesized[0]["Teff"]))



In [ ]:
ex_synth_indx = synthesized[0]["synth_index"]
ex_synth = synthesized[0]["synth_spec"]

nana_hash = temp_binning_data[1]["NANA_HASH"]
rest_fluxes = temp_binning_data[1]["flux"]
rest_ivars = temp_binning_data[1]["ivar"]
print(rest_fluxes[ex_synth_indx].shape)
print(ex_synth.shape)
print(nana_hash.shape)
moments, moments_errs = get_moment_vals(loglam, rest_fluxes, rest_ivars, ex_synth_indx, ex_synth, all_lines)

In [ ]:
print(moments.shape, moments_errs.shape)
print(synthesized.items())

In [ ]:
print(len(synthesized))

In [ ]:
def get_moment_vals_trial(log_lambda, lines, temp_dict, synth_dict, mod_list, synth_object):

    #What I believe the pseudo code to be
    #for each label in the synthesized dictionary 
    total_sum = 0
    for label in range(len(temp_dict)): #this is length 6 for the 6 cases
        #get the rest_fluxes and the rest_ivars
        rest_fluxes = temp_dict[label]["flux"]
        rest_ivars = temp_dict[label]["ivar"]
        n_hash = temp_dict[label]["NANA_HASH"]
        print("rest flux, rest ivar, and nana hash shape for temp cut", rest_fluxes.shape, rest_ivars.shape, n_hash.shape)
        
        for m in mod_list:
            temp = synth_dict[label, m]["temp"]
            print("this is for temp", temp)
            synth_indx = synth_dict[label, m]["synth_index"]
            print(f"for mod {m} and exclusive temp bin {temp}, there are {synth_indx.sum()} number of spectra")
            total_sum+=synth_indx.sum()
            print()

    print("Total number of synthesized spectra which is", total_sum, "should be equal to the num of spectra which is", len(temp_dict[0]["flux"]))
    assert False
            
    

    #loop through the temps, and hashs at that temp
        #at the same label got the synth_index and the synth_spec
            #get the moments and moments_errs
                #grab the rows that match NANA HASH, add the relevant things
    
    delta_log_lambda_pixel = get_delta_log_lam(log_lambda)
    N, M = rest_flxes[synth_indx].shape


    lam = 10 ** log_lambda

    exponents = np.arange(5)
    moments = np.zeros((N, len(lines), len(exponents))) + np.nan
    moment_errs = np.zeros((N, len(lines), len(exponents))) + np.nan

    for j, (line, wid) in enumerate(zip(lines, half_wids)):
        logline = np.log10(line)
        integration_weight = delta_log_lambda_pixel * LN10 * lam * (np.abs(logline - log_lambda) < wid)
        resid = rest_flxes[synth_indx] - synth[synth_indx]
        diff = lam - line #this is hogg's lam-lam0 he wrote about
    
        #compute vals and corresponding errors
        for k, expo in enumerate(exponents):
            something = integration_weight * diff ** expo
            moments[:, j, k] = np.sum(resid * something[None, :], axis=1)
            moment_errs[:, j, k] = np.sqrt(np.sum(something[None, :] ** 2 / rest_ivrs[synth_indx], axis=1))

    return moments, moment_errs

In [ ]:
get_moment_vals_trial(loglam, all_lines, temp_binning_data, synthesized, Mod_list)

In [ ]:
synth_dict[0]["synth_index"]

In [ ]:
#For temp binning, return the indices that correlate with each nested bin
#and use the mod immediately and perhaps return the indices for training and synthesizing

#create the synth object, and return the 

In [ ]:
temp_cuts

In [ ]:
def hogg_synth(mod_list, t_cuts, model_dict):

    synth = np.zeros_like(data) + np.nan
    for mod in mod_list:
        for cut in t_cuts:
            model = model_dict([mod, cut])
            idx = 